## Import the required libraries 

In [3]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier

## Load Data & Apply Preprocessor

In [ ]:
# Load clean data
df = pd.read_csv("../data/processed/clean_churn.csv")

# Separate features and target
y = df['churn'].apply(lambda x: 1 if str(x).strip().lower() == 'yes' else 0)
X = df.drop(columns=['churn'])

# Train-test split (same as in models/preprocessor.joblib)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Load the preprocessor artifact saved in models/preprocessor.joblib
preprocessor = joblib.load("../models/preprocessor.joblib")

# Transform datasets
X_train_transformed = preprocessor.transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

print("Train shape:", X_train_transformed.shape)
print("Test shape:", X_test_transformed.shape)

Train shape: (5625, 45)
Test shape: (1407, 45)


## Train and Evaluate Models 

In [4]:
# Calculate scale_pos_weight dynamically for XGBoost based on class imbalance
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_weight = neg_count / pos_count

# Define candidate models dictionary
candidates = {
    "logistic regression":     (LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42), False),
    "decision tree (depth 5)": (DecisionTreeClassifier(max_depth=5, class_weight="balanced", random_state=42), False),
    "random forest":           (RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42), False),
    "gradient boosting":       (GradientBoostingClassifier(random_state=42), False),
    "k-nearest neighbours":    (KNeighborsClassifier(n_neighbors=25), False),
    "xgboost":                 (XGBClassifier(scale_pos_weight=scale_weight, eval_metric='logloss', random_state=42), False)
}

# Run the loop to train and evaluate all models
rows = []
trained_models = {}

for name, (estimator, needs_scaling) in candidates.items():
    # Fit model
    estimator.fit(X_train_transformed, y_train)
    
    # Save reference to trained estimator
    trained_models[name] = estimator
    
    # Evaluate ROC-AUC
    rows.append({
        "model": name,
        "train AUC": roc_auc_score(y_train, estimator.predict_proba(X_train_transformed)[:, 1]),
        "test AUC": roc_auc_score(y_test, estimator.predict_proba(X_test_transformed)[:, 1]),
    })

# Display comparison table ranked by test AUC
comparison = pd.DataFrame(rows).set_index("model").round(3)
comparison["gap"] = (comparison["train AUC"] - comparison["test AUC"]).round(3)
print(comparison.sort_values(by="test AUC", ascending=False).to_string())

                         train AUC  test AUC    gap
model                                              
gradient boosting            0.883     0.839  0.044
logistic regression          0.851     0.835  0.016
decision tree (depth 5)      0.855     0.828  0.027
k-nearest neighbours         0.858     0.826  0.032
random forest                1.000     0.815  0.185
xgboost                      0.989     0.811  0.178


- Even though gradient boosting ranked 1, I will go with logistic regression (ranked 2) because it had a train-test gap of only 0.016 (compared to Gradient Boosting's 0.044).
- Performance is also very much similar. 

## Further Analysis of the chosen mmodel ( Logistic Regression )

In [5]:
# Get Logistic Regression model from your dictionary
lr_model = trained_models["logistic regression"]

y_pred_lr = lr_model.predict(X_test_transformed)

print("--- Logistic Regression Classification Report ---")
print(classification_report(y_test, y_pred_lr))

print("--- Logistic Regression Confusion Matrix ---")
print(confusion_matrix(y_test, y_pred_lr))

--- Logistic Regression Classification Report ---
              precision    recall  f1-score   support

           0       0.90      0.70      0.79      1033
           1       0.49      0.80      0.61       374

    accuracy                           0.73      1407
   macro avg       0.70      0.75      0.70      1407
weighted avg       0.79      0.73      0.74      1407

--- Logistic Regression Confusion Matrix ---
[[723 310]
 [ 76 298]]


- It has an outstanding recall. 